In [1]:
import json
import typing
import ollama
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.runnables.graph import MermaidDrawMethod
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langgraph.graph import StateGraph, END
from pydantic import BaseModel 
from pprint import pprint
from IPython.display import Image, display

## Defining the open-source LLM with Ollama

In [7]:
llm_opensource = "llama3.2"

In [14]:
q = "Who are the top 10 most winners of the Premier League?"

q = "Who are the top 10 latest winners of Premier League?"

In [15]:
result = ollama.chat(model = llm_opensource, 
                        messages = [{"role":"system", "content":""},
                                    {"role":"user", "content":q}])

In [16]:
print(result.message.content)

Here are the top 10 latest winners of the Premier League, from the most recent to the least recent:

1. 2022-23 season - Manchester City
2. 2021-22 season - Manchester City
3. 2020-21 season - Manchester City
4. 2019-20 season - Liverpool
5. 2018-19 season - Manchester City
6. 2017-18 season - Manchester City
7. 2016-17 season - Chelsea
8. 2015-16 season - Leicester City
9. 2014-15 season - Chelsea
10. 2013-14 season - Manchester City

Note: The 2020-21 season was completed behind closed doors due to the COVID-19 pandemic.


## Defining the Tools

In the LangGraph context, tools are external functions that the graph nodes can call to execute specific tasks, such as search in the web, interact with APIs and manipulate data. They allow the LangGraph conversation flow access external funcionalities, becoming the agent more dynamical and able to handle with different kind of requests.

In [17]:
@tool("search_tool")
def search_tool(q:str) -> str:
    """Search on DuckDuckGo navigator using `q` as input"""
    return DuckDuckGoSearchRun().run(q)

In [18]:
# Testing
print(search_tool.invoke(q))

4 weeks ago -Only five clubs have won the title in three consecutive seasons:Huddersfield Town, Arsenal, Liverpool, Manchester United (twice) and Manchester City, with the latter being the only club to have won four successive titles. ... 24 clubs which have won the English top level title, including 7 ... 1 day ago -Twenty clubs are competing in the 2025–26 season – top seventeen from the previous season and three promoted from the Championship. Leicester City, Ipswich Town and Southampton were relegated to the EFL Championship for the 2025–26 season, whilstLeeds United, Burnley, and Sunderland, as ... 5 days ago -After Arsenal led the division for much of the 2007–08 season,Manchester Unitedretained the championship on the final day of the season and won their eleventh Premier League title in 2008–09 after much competition from Liverpool. Chelsea reclaimed the league in 2009–10, scoring a record ... 1 week ago -Premier League · 20 20 13 10 9 7 6 6 4 4 3 3 3 3 2 2 2 2 2 1 1 1 1 1 · To

In [19]:
@tool("final_answer_tool")
def final_answer_tool(text:str) -> str:
    """
    Return a natural language answer to user using the `input text`.
    You must provide the maximum of possible context and specify the source of the information.
    """

    return text

In [21]:
# Testing
print(final_answer_tool.invoke("Test") )

Test


In [23]:
# Tools dict
dic_tools = {"search_tool":search_tool, 
             "final_answer_tool":final_answer_tool}

## Adding tools to help the decision-making processo of the Agent

In [24]:
prompt = """
You are a specialist in news, you must answer all the questions of the user, you can use the tools list provided to you.
Your goal is provide the best possible answer to the user, including important informations about the sources and tools used.

Look, when you call a tool, you need to provide the name of the tool and the argument that it uses in JSON format.
For each call, you MUST use ONLY on tool and the answer format ALWAYS NEEDS to follow this patter:
```json
{"name": "<tool_name>", "parameters": "<tool_input_key>":"<tool_input_value>"}
```

You must remember, You MUST NOT use the tool with the same query more than once.
You must remember, if the user doesn't make a specific question, you MUST use the tool `final_answer_tool` directly.

Everytime the user ask you a question, you need to store in your memory.
Everytime you find some information related to user question, you have to store in your memory.

You must try collect informations from several sources before providing a answer to the user.
After collect enough informations to answer the user question, use the tool `final_answer_tool`.
"""

In [25]:
str_tools = "\n".join([str(n+1)+". `"+str(v.name)+"`: "+str(v.description) for n,v in enumerate(dic_tools.values())])

In [26]:
print(str_tools)

1. `search_tool`: Search on DuckDuckGo navigator using `q` as input
2. `final_answer_tool`: Return a natural language answer to user using the `input text`.
You must provide the maximum of possible context and specify the source of the information.
